In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HOME"] = "/media/ubuntu/BE6A02386A01EDC9/huggingface"
os.environ['HF_HUB_CACHE'] = "/media/ubuntu/BE6A02386A01EDC9/huggingface/hub"
os.environ['HF_DATASET_HOME'] = "/media/ubuntu/BE6A02386A01EDC9/huggingface/datasets"

import timeout_decorator
from timeout_decorator import TimeoutError

import sys
sys.path.append("../../../../LoRO")
sys.path.append('..')

from utils import *

In [2]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from datasets import load_dataset
import tqdm

tokenizer = AutoTokenizer.from_pretrained("3mei/llama_3.2_3B_instruct_4bit_reflection_405_v2_8k_gsm8k_3e_qkvogud_mlab_instr_resp")
model = AutoModelForCausalLM.from_pretrained("3mei/llama_3.2_3B_instruct_4bit_reflection_405_v2_8k_gsm8k_3e_qkvogud_mlab_instr_resp")

`low_cpu_mem_usage` was None, now default to True since model is quantized.


In [3]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )
    )
    (norm): 

In [ ]:
# model = model_obfuscation(model) # need to modify the obfuscation code due to the data type. Or you can launch other float model.

In [4]:
dataset = load_dataset("GSM8K","main")['test']
print(dataset)

Dataset({
    features: ['question', 'answer'],
    num_rows: 1319
})


In [5]:
question_answerer = pipeline("text-generation", model=model, tokenizer=tokenizer)

Device set to use cuda:0


In [6]:
idx = 679
question = dataset[idx]['question']

expected_answer =  dataset[idx]['answer']

prompt = [{ "role" : "user" , "content" : '{} Please think step by step, give the final number in ONE new line after ####, without other words. Your answer will be considered wrong if not follow this rule.'}]
print(prompt)
print(expected_answer)
print(prompt[0]['content'].format(question))

[{'role': 'user', 'content': '{} Please think step by step, give the final number in ONE new line after ####, without other words. Your answer will be considered wrong if not follow this rule.'}]
First find the number of hash browns per potato: 36 hash browns / 6 potatoes = <<36/6=6>>6 hash browns/potato
Then multiply that number by the number of potatoes to find the total number of hash browns: 6 hash browns/potato * 96 potatoes = <<6*96=576>>576 hash browns
#### 576
If 6 potatoes makes 36 hash browns, how many hash browns can you make out of 96 potatoes? Please think step by step, give the final number in ONE new line after ####, without other words. Your answer will be considered wrong if not follow this rule.


In [7]:
query = prompt
query[0]['content'] = query[0]['content'].format(question)
answer = question_answerer(query, do_sample=False)
# print(answer[0]['generated_text'][1]['content'])
print(answer[0]['generated_text'][1]['content'].split('####')[-1])

/home/ubuntu/anaconda3/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/ubuntu/anaconda3/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


 576


In [8]:
timeout_seconds  = 60

@timeout_decorator.timeout(timeout_seconds, timeout_exception=TimeoutError)
def inference(pipe, query):
    return pipe(query, do_sample=False)

correct = 0
total = 0

model.eval()

progress_bar = tqdm.tqdm(range(len(dataset)))

for i in progress_bar:
    
    total += 1
    
    question = dataset[i]['question']
    expected_answer =  dataset[i]['answer'].split('#### ')[-1]
    
    query = [{ "role" : "user" , "content" : '{} Please think step by step, give the final number in a new line after #### without any other words.'.format(question)}]
    try:
        answer = inference(question_answerer, query)[0]['generated_text'][1]['content'].split('####')[-1].strip()
    except:
        print("TimeoutError index:{}".format(i))

    if answer == expected_answer:
        correct += 1
    
    progress_bar.set_postfix({'correct': correct, 'total': total, 'acc': correct/total})

print("correct:{}, total:{}, accuracy:{}".format(correct, total, correct/total))

  3%|▎         | 37/1319 [03:15<1:22:37,  3.87s/it, correct=16, total=37, acc=0.432]

TimeoutError index:36
